# Interpolasi Missing Value Dari Data Polusi Udara Kecamatan Kwanyar kab.bangkalan rentang waktu 31 agustus 2025 hingga 31 agustus 2026

In [1]:
!pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.4 MB/s eta 0:00:00


### 1. Muat Data dan Pembersihan (Outlier & Missing Values)
Tahap ini bertujuan untuk membaca data historis `KualitasUdara_Kwanyar.csv` dari tanggal 31 Agustus 2025 - 31 Agustus 2026. Nilai ekstrem (outliers) akan dideteksi menggunakan IQR lalu dihapus (diubah menjadi NaN). Setelah itu, seluruh missing values akan diisi kembali (imputasi) menggunakan metode interpolasi waktu.

In [2]:
import pandas as pd
import numpy as np

# ---------- 1. Muat dan bersihkan data ----------
# Ganti nama file sesuai dengan milikmu
df = pd.read_csv('KualitasUdara_Kwanyar.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# Paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai kosong bawaan/non-numerik: {n_missing_before}")

# Deteksi Outlier dengan IQR
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Ubah nilai outlier menjadi NaN
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

# Imputasi Missing Value & Outlier menggunakan interpolasi waktu
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

print("Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.")

# --- KODE TAMBAHAN UNTUK MENYIMPAN HASIL INTERPOLASI ---
# Kembalikan index 'date' menjadi kolom agar format CSV rapi
df_clean_csv = df_clean.reset_index()

# Simpan ke CSV
nama_file_bersih = 'KualitasUdara_Kwanyar_Cleaned.csv'
df_clean_csv.to_csv(nama_file_bersih, index=False)

print(f" File hasil interpolasi bernama '{nama_file_bersih}' ")


Jumlah nilai kosong bawaan/non-numerik: 173
Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.
 File hasil interpolasi bernama 'KualitasUdara_Kwanyar_Cleaned.csv' 


### 2. Imputasi Missing Value dengan Interpolasi Waktu
Dalam analisis deret waktu (time series), data yang hilang (missing values) atau data outlier yang telah dihapus (diubah menjadi NaN) tidak boleh dibiarkan kosong karena akan menggagalkan ekstraksi fitur TSFEL. Untuk mengatasinya, kita menggunakan metode **Interpolasi Waktu (Time Interpolation)**.

Metode ini memperkirakan nilai yang hilang dengan menarik garis lurus imajiner antara titik data sebelum dan sesudah nilai yang kosong, dengan mempertimbangkan secara proporsional jarak waktu antar titik tersebut.

**Rumus Dasar (Linear Interpolation):**
$$y = y_0 + (x - x_0) \frac{y_1 - y_0}{x_1 - x_0}$$

**Keterangan:**
* $y$ = Nilai polutan yang dicari (diimputasi) pada waktu $x$.
* $y_0$ = Nilai polutan valid yang tercatat sebelum data kosong.
* $y_1$ = Nilai polutan valid yang tercatat setelah data kosong.
* $x$ = Waktu (tanggal) dari data yang kosong.
* $x_0$ = Waktu (tanggal) valid sebelum data kosong.
* $x_1$ = Waktu (tanggal) valid sesudah data kosong.

**Contoh Perhitungan Manual (Sesuai Data Asli NO2 Kwanyar):**
Berdasarkan data mentah satelit, perekaman NO2 gagal (kosong) pada tanggal **10 September 2025**. Kita akan menghitung nilai penggantinya secara manual menggunakan data hari sebelum (9 Sept) dan sesudahnya (11 Sept).

**Tabel Sebelum Imputasi:**

| Tanggal ($x$) | Konsentrasi NO2 ($y$) | Keterangan |
| :--- | :--- | :--- |
| 9 Sept 2025 ($x_0$) | 0.000028185 ($y_0$) | Data Valid |
| 10 Sept 2025 ($x$) | NaN | **Missing Value** |
| 11 Sept 2025 ($x_1$) | 0.000010713 ($y_1$) | Data Valid |

Karena selisih waktu dari $x_0$ ke $x$ adalah 1 hari, dan rentang waktu dari $x_0$ ke $x_1$ adalah 2 hari, maka perhitungannya:

$$y = 0.000028185 + (1) \frac{0.000010713 - 0.000028185}{2}$$

$$y = 0.000028185 + \frac{-0.000017472}{2}$$

$$y = 0.000028185 - 0.000008736$$

$$y = 0.000019449$$

**Tabel Setelah Imputasi:**

| Tanggal | Konsentrasi NO2 | Keterangan |
| :--- | :--- | :--- |
| 9 Sept 2025 | 0.000028185 | Data Valid (Tampil di Excel sebagai 2.82E-05) |
| 10 Sept 2025 | **0.000019449** | **Hasil Interpolasi (Tampil di Excel sebagai 1.94E-05)** |
| 11 Sept 2025 | 0.000010713 | Data Valid (Tampil di Excel sebagai 1.07E-05) |

Metode ini diimplementasikan secara otomatis di dalam kode menggunakan fungsi `df.interpolate(method='time')`, yang akan menyapu bersih seluruh baris kosong sepanjang tahun dan mengisinya dengan estimasi tren waktu yang akurat. Fungsi tambahan `.ffill().bfill()` digunakan sebagai pengaman jika data kosong berada di urutan paling awal atau paling akhir (di mana tidak ada data pengepit).

### 3. Ekstraksi Fitur Polutan (TSFEL)
Melakukan ekstraksi data deret waktu polutan $NO_2$ menjadi 68 fitur statistik, spektral, dan temporal yang diminta menggunakan library TSFEL.

In [3]:
import inspect
import tsfel.feature_extraction.features as tsfel_features

# Frekuensi sampling (1 per hari)
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- Daftar 68 fitur PERSIS seperti instruksi ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))

def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")

# Menampilkan pratinjau data hasil ekstraksi
print("\nCuplikan hasil ekstraksi fitur:")
display(extracted_features_final.head())

# Menyimpan file ke dalam format CSV di lingkungan Colab
nama_file_csv = f'{target_pollutant}_Kwanyar_TSFEL.csv'
extracted_features_final.to_csv(nama_file_csv, index=False)
print(f"  File {nama_file_csv} sudah tersimpan .")

Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68

Cuplikan hasil ekstraksi fitur:


,abs_energy,auc,autocorr,average_power,calc_centroid,calc_max,calc_mean,calc_median,calc_min,calc_std,...,spectral_spread,spectral_variation,spectrogram_mean_coeff,sum_abs_diff,wavelet_abs_mean,wavelet_energy,wavelet_entropy,wavelet_std,wavelet_var,zero_cross
0,3.180175e-07,0.009898,11.0,8.712809e-10,177.776064,0.000055,0.000027,0.000025,-0.000004,0.000012,...,0.151269,0.368722,2.142332e-10,0.001943,0.000001,0.000016,2.099813,0.000016,2.881428e-10,6.0


  File NO2_Kwanyar_TSFEL.csv sudah tersimpan .


In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# 1. Baca data ekstraksi fitur gabungan 1 kelas
df_all = pd.read_csv("ekstraksi_fitur_no2_psd-a.csv")

# 2. Pisahkan kolom identitas dan kolom fitur (68 fitur)
kolom_fitur = df_all.columns[3:]
fitur_mentah = df_all[kolom_fitur].fillna(0)

# 3. Standardisasi data agar perhitungannya adil
scaler = StandardScaler()
fitur_scaled = scaler.fit_transform(fitur_mentah)

# 4. Cari indeks datamu secara otomatis (Mencari namamu: Fahdimas)
index_kwanyar = df_all[df_all['nama'].str.contains('Fahdimas', case=False, na=False)].index[0]

# 5. Hitung tingkat kemiripan (Cosine Similarity) Kwanyar dengan seluruh data
kwanyar_vector = fitur_scaled[index_kwanyar].reshape(1, -1)
skor_kemiripan = cosine_similarity(kwanyar_vector, fitur_scaled)[0]

# 6. Masukkan skor ke dalam dataframe, ubah ke persen
df_all['Skor_Kemiripan (%)'] = (skor_kemiripan * 100).round(2)

# --- REVISI DI SINI ---
# Buang HANYA datamu sendiri agar tidak muncul 100% (self-match). Data Umam biarkan masuk.
df_filter = df_all.drop(index=index_kwanyar).copy()

# Urutkan dari skor terbesar
top_kembar = df_filter.sort_values(by='Skor_Kemiripan (%)', ascending=False)

# 7. Tampilkan hasil
print("--- 5 DATA DENGAN POLA POLUSI NO2 PALING MIRIP ---")
display(top_kembar[['nama', 'daerah', 'Skor_Kemiripan (%)']].head(5).reset_index(drop=True))

--- 5 DATA DENGAN POLA POLUSI NO2 PALING MIRIP ---


,nama,daerah,Skor_Kemiripan (%)
0,M. Fatihul Umam,"Kwanyar, Bangkalan",78.13
1,Kurnia Maulinda Sari,"Labang, Bangkalan",60.11
2,Triswanti Jannatul Ma'wa,Kedungpring Lamongan,57.36
3,Nurhabibatul Umah,Kerek Tuban,53.70
4,Ahmad Maulana Ishaq,Bandarkedungmulyo,52.36
